# Setup Vector Database for Retrieval Evaluation
## Dataset: YuITC/Vietnamese-Legal-Documents

This notebook setup a vector database for evalutating between different embedding models in Vietnamese.

### Dataset Structure

The dataset consists of 2 split: `train` and `test`. Below is the structure of the dataset:

| Column | Description |
|---|---|
| `question` | Natural language query (Vietnamese legal question) |
| `context_list` | List of relevant document passages (ground-truth answers) |
| `qid` | Unique query ID |
| `cid` | List of corpus document IDs that are relevant |

### How to initialize a vector database
1. **Create Corpus**: All unique passages from `context_list` across the dataset are pooled into one corpus.
2. **Setup Vector DB**: Create a consistant client on file systems for testing and evaluating purpose.
3. **Store**: Store the corpus using different embedding models.
4. **Verify**: Check the stored corpus by some simple similarity search (not for evaluation yet).

## 1. Setup environment, libraries

Install necessary dependancies and check cuda/mps availability.

In [ ]:
%pip install sentence-transformers datasets pandas numpy matplotlib seaborn pyvi tqdm chromadb -q

In [1]:
import torch
print(f"PyTorch version : {torch.__version__}")
# Default to CPU
device = "cpu"
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    device = "cuda"
# print(f"MPS available   : {torch.backends.mps.is_available()}")
# if torch.backends.mps.is_available():
#     print(f"GPU             : MPS")
#     device = "mps"

PyTorch version : 2.11.0
CUDA available  : False


## 2. Load the Dataset

Load the Dataset for processing, inspect some metadata and some examples of an instance inside. In this evaluation, we use `test` split for faster processing and less burden for hardware.

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np

print("Loading YuITC/Vietnamese-Legal-Documents ...")
raw = load_dataset("YuITC/Vietnamese-Legal-Documents")
print(raw)

# Use the test split for evaluation; fall back to train if test is absent
split = "test" if "test" in raw else list(raw.keys())[0]
df = raw[split].to_pandas()

print(f"\nUsing split : '{split}'")
print(f"Rows        : {len(df):,}")
print(f"Columns     : {df.columns.tolist()}")
df.head(3)

/Users/thanhdat/Workspace/Dat_all_Mac_projects/Demo/vi_embed_eva/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading YuITC/Vietnamese-Legal-Documents ...


DatasetDict({
    train: Dataset({
        features: ['question', 'context_list', 'qid', 'cid'],
        num_rows: 89261
    })
    test: Dataset({
        features: ['question', 'context_list', 'qid', 'cid'],
        num_rows: 29746
    })
})

Using split : 'test'
Rows        : 29,746
Columns     : ['question', 'context_list', 'qid', 'cid']


,question,context_list,qid,cid
0,Phó Tổng Giám đốc Ngân hàng Chính sách xã hội ...,[Áp dụng chế độ tiền lương và phụ cấp quy định...,70867,[140864]
1,Ai có thẩm quyền quyết định thành lập Hội đồng...,[Thành lập Hội đồng\n1. Bộ trưởng Bộ Y tế ra q...,813,[62339]
2,Thời hiệu xử phạt đối với nhà xuất bản thực hi...,[Điều 5. Thời hiệu xử phạt vi phạm hành chính\...,40392,[63171]


## 3. Setup Vector Database

In [3]:
import chromadb

client = chromadb.PersistentClient(path="./database")

## 4. Build corpus

Retrieve unique documents from `context_list` for storing inside vector database.

In [4]:
corpus = {}          # list of unique passage texts (our search index)

for r_idx, data in df.iterrows():
    # print(data["context_list"])
    for p_idx , passage in enumerate(data["context_list"]):
        p = passage.strip()
        pid = f"{data['cid'][p_idx]}"
        if p and pid not in corpus:
            corpus[pid] = p

print(f"Total unique corpus passages : {len(corpus):,}")

Total unique corpus passages : 23,907


## 5. Define embedding Model

In [5]:
MODELS = [
    # {
    #     "name": "AITeamVN/Vietnamese_Embedding",
    #     "label": "AITeamVN (BGE-M3)",
    #     # BGE-M3 fine-tuned on 300K Vietnamese triplets; 2048-token context
    # },
    {
        "name": "dangvantuan/vietnamese-document-embedding",
        "label": "VN-DocEmbed",
        "trust_remote_code": True,
        # Best for long passages; 8096-token context window
    },
    # {
    #     "name": "Alibaba-NLP/gte-multilingual-base",
    #     "label": "Alibaba-GTE",
    #     "trust_remote_code": True,
    #     # Multilingual high-performance model in General Text Embedding
    # },
    # {
    #     "name": "BAAI/bge-m3",
    #     "label": "BGE-M3 (multilingual)",
    #     # General multilingual baseline
    # },
]

print(f"Models to evaluate: {len(MODELS)}")
for m in MODELS:
    print(f"  • {m['label']}")

Models to evaluate: 1
  • VN-DocEmbed


## 6. Store document corpus for different embedding models

Iterate each models to define embedding function for database collection. Add document corpus into the respective collection.

In [8]:
import time
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

print("STORING")
for model in MODELS:
    print(f"\n{'='*65}")
    print(f"  {model['label']}  ({model['name']})")
    print(f"{'='*65}")
    
    trust_kargs = {'trust_remote_code': True if model["trust_remote_code"] else False}
    try:
        collection = client.create_collection(
            name=model["label"],
            embedding_function=SentenceTransformerEmbeddingFunction(
                model_name=model["name"],
                device=device,
                **trust_kargs
            )
        )
    except Exception as e: 
        collection = client.get_collection(model['label'])
        
    collection.add(
        ids=["ids"],
        documents=["test chút thôi"] 
    )
    
    # Add document into database with retries and error handling
    # for cid, context in corpus.items():
    #     for attempt in range(3):  # Try up to 3 times
    #         try:
    #             # Operation that might fail
    #             collection.add(
    #                 ids=[cid],
    #                 documents=[context]
    #             )
    #             break  # Exit the loop on success
    #         except Exception as e:
    #             print(f"Attempt {attempt + 1} failed: {e}. Retrying...")
    #             time.sleep(2)  # Wait before retrying
    #     else:
    #         print("Failed to add documents after 3 attempts. Skipping this document.")
        
      

STORING

  VN-DocEmbed  (dangvantuan/vietnamese-document-embedding)


IndexError: index 4326137488 is out of bounds for dimension 0 with size 5 in add.

## 7. Verify database

Check simple semantic search on the newly created collections.

In [ ]:
print("VERIFYING")
for model in MODELS:
    collection = client.get_collection(model["label"])
    retrieve_result = collection.query(
        query_texts=["Liên đoàn Luật sư Việt Nam là tổ chức xã hội - nghề nghiệp có tư cách pháp nhân, có con dấu, tài khoản riêng?"],
        n_results=10,
    )

    for i, doc in enumerate(retrieve_result["documents"][0]):
        print(f"Document {i + 1}:")
        # print("Name:", retrieve_result["metadatas"][0][i].get("name", "N/A"))
        print("CID:", retrieve_result["ids"][0][i])
        print("Content:", doc)
        # print("Id:", retrieve_result["ids"][0][i])
        # print("Document number:", retrieve_result["metadatas"][0][i].get("numberDoc", "N/A"))
        # print("Fields:", retrieve_result["metadatas"][0][i].get("fields", "N/A"))
        # print("Metadata:", retrieve_result["metadatas"][0][i])
        print("Score:", retrieve_result["distances"][0][i])
        print("-"* 50)  # Separator for readability

## 8. Delete collections (Optional)

In [7]:
for model in MODELS:
    client.delete_collection(model["label"])